## Load packages

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
import os
from UniProtMapper import ProtMapper
# get project root (two levels up from this notebook)
project_root = os.path.abspath(os.path.join(os.path.dirname('src'), '..'))
# if in notebook:
# project_root = os.path.abspath('..')   # or adjust as needed

if project_root not in sys.path:
    sys.path.insert(0, project_root)
from importlib import reload

/mnt/aiongpfs/users/adhal/micromamba/envs/scrna_target_idf/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Load scHIIT packages

In [2]:
import src.methods.schiit_method.tf_filters.base_filter as schiit_base
import src.methods.schiit_method.tf_filters.jsd_filter as schiit_main
import src.methods.schiit_method.network.grn_base as schiit_grn
reload(schiit_main)
reload(schiit_grn)

<module 'src.methods.schiit_method.network.grn_base' from '/mnt/lscratch/users/adhal/SingleCellUtils/src/methods/schiit_method/network/grn_base.py'>

##  Load data method creation for query + reference data 

- reference data: single nuclei data
- query data: seattle AD data

In [3]:
ref_data_path = "/mnt/lscratch/users/adhal/SingleCellUtils/data/scHIIT_ref/ref_nuclei_231125.h5ad"
query_data_path = "/mnt/lscratch/users/adhal/SingleCellUtils/data/scHIIT_query/sead_sampled_10K_171125.h5ad"
ref_adata = sc.read_h5ad(ref_data_path)
query_adata = sc.read_h5ad(query_data_path)

In [4]:
query_adata.obs['Class'].unique()

['Neuronal: Glutamatergic', 'Neuronal: GABAergic']
Categories (2, object): ['Neuronal: GABAergic', 'Neuronal: Glutamatergic']

In [5]:
## Function to load from h5ad and harmonize the observation columns

In [6]:
ref_adata

AnnData object with n_obs × n_vars = 163412 × 22298

In [7]:
query_adata

AnnData object with n_obs × n_vars = 17937 × 36412
    obs: 'assay_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'is_primary_data', 'donor_id', 'Neurotypical reference', 'Class', 'Subclass', 'Supertype', 'Age at death', 'Years of education', 'Cognitive status', 'ADNC', 'Braak stage', 'Thal phase', 'CERAD score', 'APOE4 status', 'Lewy body disease pathology', 'LATE-NC stage', 'Microinfarct pathology', 'Specimen ID', 'PMI', 'Number of UMIs', 'Genes detected', 'Fraction mitochrondrial UMIs', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'ADNC_colors', 'APOE4 status_colors', 'Age at death_colors'

In [8]:
"""
Merge query (Seattle AD) and reference AnnData when reference has no metadata.

Reference data: Just a raw count matrix (163412 × 22298)
Query data: Full Seattle AD metadata
"""

import anndata as ad
import pandas as pd
import numpy as np
import scanpy as sc
from scipy.sparse import issparse, csr_matrix

def fix_query_var_names(adata):
    neuronal_tier1 = adata.copy()
    # Save original Ensembl IDs
    neuronal_tier1.var['ensembl_id'] = neuronal_tier1.var_names.copy()
    
    # Convert categorical to string
    neuronal_tier1.var['feature_name'] = neuronal_tier1.var['feature_name'].astype(str)
    
    # Set var_names to gene symbols
    neuronal_tier1.var_names = neuronal_tier1.var['feature_name'].values
    
    # Make unique (handles the 28 duplicates)
    neuronal_tier1.var_names_make_unique()
    
    print("\nNew var_names (first 5):")
    print(neuronal_tier1.var_names[:5])
    
    print("\nDuplicate genes that got suffix:")
    duplicated_genes = neuronal_tier1.var_names[neuronal_tier1.var_names.str.contains('-[0-9]$', regex=True)]
    print(duplicated_genes[:10])
    return neuronal_tier1
    
def merge_query_reference_simple(
    query_adata: ad.AnnData,
    reference_adata: ad.AnnData,
    query_cluster_key: str = 'Subclass',
    batch_key: str = 'dataset'
) -> ad.AnnData:
    """
    Merge query and reference when reference has NO metadata columns.
    
    This is the simple case - reference is just a background matrix.
    
    Parameters
    ----------
    query_adata : AnnData
        Seattle AD with full metadata (17937 × 36412)
    reference_adata : AnnData
        Tabula Sapiens with no .obs columns (163412 × 22298)
    query_cluster_key : str
        Column in query for cell types ('Class', 'Subclass', or 'Supertype')
    batch_key : str
        Name for new column identifying query vs reference
        
    Returns
    -------
    merged_adata : AnnData
        Merged object ready for benchmarking
    """
    
    print("="*60)
    print("MERGING QUERY + REFERENCE (Reference has no metadata)")
    print("="*60)
    
    # Step 1: Find common genes
    print("\nStep 1: Finding common genes...")
    common_genes = query_adata.var_names.intersection(reference_adata.var_names)
    print(f"  Query genes:     {query_adata.n_vars:,}")
    print(f"  Reference genes: {reference_adata.n_vars:,}")
    print(f"  Common genes:    {len(common_genes):,}")
    
    if len(common_genes) == 0:
        raise ValueError("No common genes found! Check gene naming (symbols vs ENSEMBL IDs)")
    
    # Subset to common genes
    query_subset = query_adata[:, common_genes].copy()
    reference_subset = reference_adata[:, common_genes].copy()
    
    # Step 2: Add minimal metadata to reference
    print("\nStep 2: Adding metadata...")
    
    # Query: add dataset identifier
    query_subset.obs[batch_key] = 'query'
    query_subset.obs['cluster'] = query_subset.obs[query_cluster_key].astype(str)
    
    # Reference: create minimal metadata
    reference_subset.obs[batch_key] = 'reference'
    reference_subset.obs['cluster'] = 'background'  # All reference cells = background
    
    # Add the query cluster key column to reference (with 'background' values)
    reference_subset.obs[query_cluster_key] = 'background'
    
    print(f"  Query clusters: {query_subset.obs['cluster'].nunique()}")
    print(f"  Reference: all labeled as 'background'")
    
    # Step 3: Handle query-specific metadata columns
    print("\nStep 3: Preserving query metadata...")
    query_only_cols = set(query_subset.obs.columns) - set(reference_subset.obs.columns)
    print(f"  Query-specific columns: {len(query_only_cols)}")
    
    # Add these columns to reference with NA values
    for col in query_only_cols:
        dtype = query_subset.obs[col].dtype
        if dtype == 'object' or dtype.name == 'category':
            reference_subset.obs[col] = pd.Categorical(['NA'] * reference_subset.n_obs)
        elif dtype in ['float64', 'float32', 'int64', 'int32']:
            reference_subset.obs[col] = np.nan
        else:
            reference_subset.obs[col] = 'NA'
    
    # Step 4: Ensure both have same var columns
    print("\nStep 4: Aligning gene metadata...")
    if query_subset.var.shape[1] > 0:
        # Query has gene metadata, add to reference
        for col in query_subset.var.columns:
            if col not in reference_subset.var.columns:
                reference_subset.var[col] = 'NA'
    
    # Step 5: Merge
    print("\nStep 5: Concatenating datasets...")
    merged_adata = ad.concat(
        [query_subset, reference_subset],
        axis=0,
        join='outer',
        label='batch_source',
        keys=['query', 'reference'],
        index_unique='-',
        merge='same'
    )
    
    # Verify batch_key exists
    if batch_key not in merged_adata.obs.columns:
        merged_adata.obs[batch_key] = merged_adata.obs['batch_source']
    
    print("\n" + "="*60)
    print("MERGE COMPLETE")
    print("="*60)
    print(f"Total cells:      {merged_adata.n_obs:,}")
    print(f"Total genes:      {merged_adata.n_vars:,}")
    print(f"Query cells:      {(merged_adata.obs[batch_key] == 'query').sum():,}")
    print(f"Reference cells:  {(merged_adata.obs[batch_key] == 'reference').sum():,}")
    
    print("\nQuery cluster distribution:")
    query_clusters = merged_adata[merged_adata.obs[batch_key] == 'query'].obs['cluster'].value_counts()
    for cluster, count in query_clusters.items():
        print(f"  {cluster}: {count:,} cells")
    
    return merged_adata


def verify_merge(merged_adata: ad.AnnData, batch_key: str = 'dataset'):
    """
    Verify the merge worked correctly.
    """
    print("\n" + "="*60)
    print("VERIFICATION")
    print("="*60)
    
    # Check basic structure
    print("\n1. Basic structure:")
    print(f"   Total cells: {merged_adata.n_obs:,}")
    print(f"   Total genes: {merged_adata.n_vars:,}")
    print(f"   Sparse matrix: {issparse(merged_adata.X)}")
    
    # Check dataset split
    print("\n2. Dataset distribution:")
    print(merged_adata.obs[batch_key].value_counts())
    
    # Check cluster key
    print("\n3. Cluster key:")
    print(f"   Column exists: {'cluster' in merged_adata.obs.columns}")
    print(f"   Unique clusters: {merged_adata.obs['cluster'].nunique()}")
    print(f"   Query clusters: {merged_adata[merged_adata.obs[batch_key]=='query'].obs['cluster'].nunique()}")
    
    # Check query metadata preservation
    print("\n4. Query metadata preservation:")
    query_specific_cols = ['Class', 'Subclass', 'Supertype', 'Cognitive status', 'Braak stage']
    for col in query_specific_cols:
        if col in merged_adata.obs.columns:
            n_query_with_data = merged_adata[
                merged_adata.obs[batch_key] == 'query'
            ].obs[col].notna().sum()
            print(f"   {col}: {n_query_with_data:,} query cells with data ✓")
        else:
            print(f"   {col}: NOT FOUND ✗")
    
    # Check for any issues
    print("\n5. Sanity checks:")
    
    # All reference cells should be 'background' cluster
    ref_clusters = merged_adata[merged_adata.obs[batch_key] == 'reference'].obs['cluster'].unique()
    if len(ref_clusters) == 1 and ref_clusters[0] == 'background':
        print("   Reference cells all labeled 'background': ✓")
    else:
        print(f"   WARNING: Reference has {len(ref_clusters)} clusters: {ref_clusters}")
    
    # No NaN in critical columns
    critical_cols = [batch_key, 'cluster']
    all_good = True
    for col in critical_cols:
        if merged_adata.obs[col].isna().any():
            print(f"   WARNING: NaN values in '{col}' column")
            all_good = False
    
    if all_good:
        print("   No NaN in critical columns: ✓")
    
    print("\n" + "="*60)
    print("Verification complete!")
    print("="*60)




In [9]:
query_adata_fixed = fix_query_var_names(query_adata)


New var_names (first 5):
Index(['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112'], dtype='object')

Duplicate genes that got suffix:
Index(['NKX3-2', 'NKX2-3', 'H1-3', 'H1-1', 'NKX2-4', 'NKX2-2', 'NKX2-8',
       'NKX2-1', 'ERVK3-1', 'NKX6-2'],
      dtype='object')


In [10]:
# ============================================================================
# USAGE
# ============================================================================
# STEP 1: Merge
merged_adata = merge_query_reference_simple(
    query_adata=query_adata_fixed,
    reference_adata=ref_adata,
    query_cluster_key='Class',  # Options: 'Class', 'Subclass', 'Supertype'
    batch_key='dataset'
)

# STEP 2: Verify
verify_merge(merged_adata, batch_key='dataset')


# STEP 4: Save
# merged_adata.write('merged_seattle_tabsap.h5ad')
print("\nMerged data ready for benchmarking! ✓")

# STEP 5: Ready to use with base class
print("\n" + "="*60)
print("USAGE WITH BENCHMARKING METHODS")
print("="*60)
del query_adata_fixed

MERGING QUERY + REFERENCE (Reference has no metadata)

Step 1: Finding common genes...
  Query genes:     36,412
  Reference genes: 22,298
  Common genes:    20,745

Step 2: Adding metadata...
  Query clusters: 2
  Reference: all labeled as 'background'

Step 3: Preserving query metadata...
  Query-specific columns: 38

Step 4: Aligning gene metadata...

Step 5: Concatenating datasets...

MERGE COMPLETE
Total cells:      181,349
Total genes:      20,745
Query cells:      17,937
Reference cells:  163,412

Query cluster distribution:
  Neuronal: GABAergic: 9,200 cells
  Neuronal: Glutamatergic: 8,737 cells

VERIFICATION

1. Basic structure:
   Total cells: 181,349
   Total genes: 20,745
   Sparse matrix: True

2. Dataset distribution:
dataset
reference    163412
query         17937
Name: count, dtype: int64

3. Cluster key:
   Column exists: True
   Unique clusters: 3
   Query clusters: 2

4. Query metadata preservation:
   Class: 17,937 query cells with data ✓
   Subclass: 17,937 query 

In [11]:
merged_adata 

AnnData object with n_obs × n_vars = 181349 × 20745
    obs: 'assay_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'is_primary_data', 'donor_id', 'Neurotypical reference', 'Class', 'Subclass', 'Supertype', 'Age at death', 'Years of education', 'Cognitive status', 'ADNC', 'Braak stage', 'Thal phase', 'CERAD score', 'APOE4 status', 'Lewy body disease pathology', 'LATE-NC stage', 'Microinfarct pathology', 'Specimen ID', 'PMI', 'Number of UMIs', 'Genes detected', 'Fraction mitochrondrial UMIs', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'dataset', 'cluster', 'batch_source'
    obsm: 'X_scVI', 'X_umap'

## Apply scHIIT

In [12]:
import numpy as np
import gc
from scipy.sparse import csr_matrix, issparse

# Create memory-safe subset (subsample background)
def create_subset(adata, target_ct, max_background=20000):
    target_mask = adata.obs['Class'] == target_ct
    background_mask = adata.obs['Class'] == 'background'
    
    target_idx = np.where(target_mask)[0]
    background_idx = np.where(background_mask)[0]
    
    if len(background_idx) > max_background:
        np.random.seed(42)
        background_idx = np.random.choice(background_idx, max_background, replace=False)
    
    keep_idx = np.concatenate([target_idx, background_idx])
    subset = adata[keep_idx, :].copy()
    
    if not issparse(subset.X):
        subset.X = csr_matrix(subset.X)
    
    return subset

# Run pipeline
reload(schiit_main)
results, stageI_II_tfs = {}, {}

tf_df = pd.read_csv('/mnt/lscratch/users/adhal/data/scrna_target_idf_v2/scrna_target_idf_v2/data/Homo_sapiens_TF.txt', sep='\t')
tf_name_list = tf_df['Symbol'].to_list()

ct = 'Neuronal: GABAergic'
adata_safe = create_subset(merged_adata, ct, max_background=100000)

neuro_top_tf = schiit_main.CoreFilterOnlyPipeline(
    adata=adata_safe,  # Use subset
    tf_list=tf_name_list,
    target_cell_type=ct,
    background_cell_type='background',
    cell_type_key='Class',
    chipseq_file=None,
    verbose=True,
    scgx_sig_file='/mnt/lscratch/users/adhal/SingleCellUtils/outputs/scGX/tier1_SEAD/sig_frames/Neuronal: GABAergic_sig_tbl.txt',
    main_filter='high_and_unique',
    jsd_method='geometric_jsd',
    top_n_high=None,
    top_jsd_pc=None,
    top_n_jsd=20,
    expr_method='scgx',
    # n_jobs=1  # Critical: single thread
)

neuro_top_tf.run()
results[ct] = neuro_top_tf.results
stageI_II_tfs[ct] = neuro_top_tf.results['core_filtered_tfs']

del adata_safe
gc.collect()

print(f"✓ Found {len(stageI_II_tfs[ct])} TFs")

  Converting sparse matrix for parallel processing...
INITIALIZED: CoreFilterOnlyPipeline
Target cell type: Neuronal: GABAergic
Target cells: 9200
Total cells: 109200
Input TFs: 1659

RUNNING: CoreFilterOnlyPipeline

[Core Filtering]
  High expression TFs: 290/1659
  Computing geometric_jsd divergence for 290 TFs...
    Using 8 processes for 290 genes...


    Parallel geometric_jsd: 100%|██████████| 290/290 [00:03<00:00, 77.15it/s]


  Top 20 TFs (geometric_jsd <= 5758.474)
  Unique expression TFs: 20/290
  Top 5 by geometric_jsd (most specific): PEG3(1616.19), MYT1L(2372.56), ZNF655(3316.37), ZNF506(4086.78), ZNF536(4286.65)
Core filtered TFs (high & unique): 20

[Additional Filtering]
  No additional filtering applied
  Final filtered TFs: 20

[Network Construction]
  No network built (baseline uses core filters only)
  Network: 20 nodes, 0 edges

[Identity TF Selection]
  Identity TFs found: 20

RESULTS SUMMARY

Filtering Cascade:
  Input TFs:        1659
  ├─ High expr:      290
  ├─ Unique expr:     20
  ├─ Core (both):     20
  └─ Final:           20

Network:
  Nodes:              20
  Edges:               0

Identity TFs:
  Found:              20

Runtime: 5.98s
✓ Found 20 TFs


## Run GRN to build transcriptional core

In [ ]:

chip_seq_file = '/mnt/lscratch/users/adhal/scrna_target_idf_v3/data/from_martin/Chip_removed_overlapped_peaks.tsv'

stage3_results = schiit_grn.run_stage_iii(
    tf_results=results[ct],
    gene_results=results[ct],
    chipseq_file=chip_seq_file,
    selection_method='largest',
    min_scc_size=2,
    output_dir='./stage3_results_20_tfs',
    verbose=True
)


STAGE III: CORE TF IDENTIFICATION

STEP 1: LOAD LITERATURE NETWORK

Loading ChIP-seq: /mnt/lscratch/users/adhal/scrna_target_idf_v3/data/from_martin/Chip_removed_overlapped_peaks.tsv
  ChIP-seq edges: 6844416

Loading CollecTRI...
  CollecTRI edges: 21699
